# Euclidean 3D (G(3,0)) — Vectors, Bivectors, Rotors

**Part I · Geometric Algebra & Core** — Tutorial 04

This tutorial is a deep dive into `BasisE3`, the Euclidean 3D algebra $G(3, 0)$. It is
the simplest and most familiar geometric algebra: three basis vectors $e_1, e_2, e_3$
that all square to $+1$, and an $8$-dimensional space of multivectors built from them.

By the end you will be able to:

- Build **vectors** and **bivectors** from coordinates and from the geometric product of
  basis vectors.
- Form bivectors with the **outer product** and read off metric relationships with the
  **inner product**.
- Construct **rotors** from angle–axis pairs (via the geometry submodule) and apply them
  with the sandwich product `R * v * ~R`.
- Compose rotations and verify the results.
- Use the **pseudoscalar** and the **Hodge dual** — recovering the cross product.
- Relate **bivectors** to **rotation planes**.

> **Prerequisites:** [Tutorial 02](../02_algebra_core/) (products, grades, reverse) and
> [Tutorial 03](../03_basis_classes/) (named blades). Rotors are created through the
> `pytanga.geometry` submodule — the same `Rotor` used across the other 3D algebras.


## 1. Setup

Three imports are needed: the `MV` type, the `BasisE3` basis class, and the geometry
dataclasses (a rotor is created and analysed through the `Geometry` convenience class).


In [1]:
import math

from pytanga import MV
from pytanga.basis import BasisE3
from pytanga.geometry import Direction, Geometry, Rotor

E3 = BasisE3()          # G(3, 0) — three basis vectors, all square to +1
geo = Geometry(E3)      # binds the algebra; OPNS/IPNS read from E3.opns (default True)


## 2. The basis blades

`BasisE3` exposes all eight blades as attributes. The names follow the same pattern as
every basis class: `e1`–`e3` for vectors, `e12`/`e23`/`e31` for bivectors, and `I` for
the pseudoscalar. (Both `e31` and its alias `e13` exist; they differ by a sign.)


In [2]:
e1, e2, e3 = E3.e1, E3.e2, E3.e3
e12, e23, e31 = E3.e12, E3.e23, E3.e31
I3 = E3.I                     # pseudoscalar e1 ∧ e2 ∧ e3

print("blades:", list(E3.blades().keys()))
print()

(e1 * e1).show("e1 * e1  (a basis vector squares to its metric value)")
(e1 * e2).show("e1 * e2  (geometric product of two basis vectors)")
(I3 * I3).show("I  * I   (the pseudoscalar squares to −1 in G(3,0))")


blades: ['e1', 'e2', 'e3', 'e12', 'e31', 'e13', 'e23', 'I']



e1 * e1  (a basis vector squares to its metric value): 1

e1 * e2  (geometric product of two basis vectors): e12

I  * I   (the pseudoscalar squares to −1 in G(3,0)): - 1

## 3. Vectors from coordinates

A vector is a grade-1 multivector. The clearest way to write one is as a string, but the
dict form is just as valid and mirrors how the coefficients are stored internally.


In [3]:
# String form — most readable for hand-written values
a = E3("1 e1 + 2 e2 + 3 e3")

# Dict form — blade name → coefficient
b = E3({"e1": 4.0, "e2": 5.0, "e3": 6.0})

a.show("a")
b.show("b")

# Read single coefficients back by blade name
print("a['e2'] =", a["e2"])
print("b['e3'] =", b["e3"])


a: e1 + 2 e2 + 3 e3

b: 4 e1 + 5 e2 + 6 e3

a['e2'] = 2.0
b['e3'] = 6.0


## 4. The geometric product of two vectors

The geometric product of two vectors is the sum of their **inner product** (a scalar) and
their **outer product** (a bivector):

$$ab = a \cdot b + a \wedge b$$

For $a = (1,2,3)$ and $b = (4,5,6)$ the scalar part is the dot product $32$ and the
bivector part is the oriented plane spanned by the two vectors.


In [4]:
a = E3("1 e1 + 2 e2 + 3 e3")
b = E3("4 e1 + 5 e2 + 6 e3")

ab = a * b
ab.show("a * b")

# Grade projection separates the two parts
print("scalar part  ⟨a b⟩₀ =", ab.grade(0), "   (= a · b)")
print("bivector part ⟨a b⟩₂ =", ab.grade(2), "   (= a ∧ b)")


a * b: 32 - 3 e12 - 6 e13 - 3 e23

scalar part  ⟨a b⟩₀ = 32    (= a · b)
bivector part ⟨a b⟩₂ = -3 e12 - 6 e13 - 3 e23    (= a ∧ b)


## 5. Bivectors: the outer product

The outer product $u \wedge v$ is an **oriented area element** — the parallelogram
spanned by $u$ and $v$. It is antisymmetric, so $v \wedge u = -\,u \wedge v$. The wedge
of two basis vectors produces the named bivectors.


In [5]:
# Wedge of basis vectors gives the three bivectors (oriented area elements)
(e1 ^ e2).show("e1 ^ e2")
(e2 ^ e3).show("e2 ^ e3")
(e3 ^ e1).show("e3 ^ e1  (= e31)")

# Antisymmetry on vectors
(e2 ^ e1).show("e2 ^ e1  (= − e1 ^ e2)")

# Outer product of two arbitrary vectors
a = E3("1 e1 + 2 e2 + 3 e3")
b = E3("4 e1 + 5 e2 + 6 e3")
(a ^ b).show("a ^ b")


e1 ^ e2: e12

e2 ^ e3: e23

e3 ^ e1  (= e31): - e13

e2 ^ e1  (= − e1 ^ e2): - e12

a ^ b: - 3 e12 - 6 e13 - 3 e23

## 6. The inner product — metric relationships

The inner product encodes the metric of the algebra. In $G(3,0)$ all basis vectors square
to $+1$, so $e_i \cdot e_j = \delta_{ij}$: orthogonal basis vectors have zero inner
product, and a vector dotted with itself gives its squared length. For grade-1 vectors the
inner product is the ordinary dot product.


In [6]:
# The inner product encodes the Euclidean metric: e_i · e_j = δ_ij
print("e1 | e1 =", e1 | e1)
print("e1 | e2 =", e1 | e2)
print("e2 | e2 =", e2 | e2)
print("e3 | e3 =", e3 | e3)

# For vectors it reduces to the ordinary dot product
a = E3("1 e1 + 2 e2 + 3 e3")
b = E3("4 e1 + 5 e2 + 6 e3")
print()
print("a | b =", a | b, "   (dot product 1·4 + 2·5 + 3·6)")


e1 | e1 = 1
e1 | e2 = 0
e2 | e2 = 1
e3 | e3 = 1

a | b = 32    (dot product 1·4 + 2·5 + 3·6)


## 7. Rotors — angle–axis

A **rotor** is an even-grade versor (scalar + bivector) that rotates via the sandwich
product $R\,v\,\tilde{R}$. Create one from an angle and an axis with the geometry
submodule's `Rotor` operator — `geo(...)` turns the dataclass into an `MV`.

For a rotation by $\theta$ about an axis $n$, the rotor is
$R = \cos(\theta/2) - \sin(\theta/2)\,B$, where $B$ is the unit bivector of the
rotation plane (the plane perpendicular to $n$). For the $z$-axis that bivector is
$e_{12}$.


In [ ]:
# 90° rotation about the z-axis
R = geo(Rotor(angle=math.pi / 2, axis=Direction(0, 0, 1)))

R.show("R = cos(θ/2) - sin(θ/2) e12")
print()
print("R * ~R =", R * ~R)          # unit rotor: the reverse is the inverse

# Apply the rotation with the sandwich (versor) product
v = E3.e1
rotated = R * v * ~R
print()
v.show("v        (e1)")
rotated.show("R v ~R   (e1 rotated 90° about z → e2)")


R = cos(θ/2) − sin(θ/2) e12: 0.7071 - 0.7071 e12


R * ~R = 1



v        (e1): e1

R v ~R   (e1 rotated 90° about z → e2): e2

## 8. Composing rotations

Rotors compose by the geometric product — the rightmost rotor is applied first. Two
half-rotations therefore give a full rotation, and the result is again a unit rotor.


In [8]:
# Two 45° rotations about z compose into one 90° rotation
R1 = geo(Rotor(angle=math.pi / 4, axis=Direction(0, 0, 1)))
R2 = geo(Rotor(angle=math.pi / 4, axis=Direction(0, 0, 1)))

R = R2 * R1      # order matters: R1 is applied first
R.show("R = R2 * R1")
print("R * ~R =", R * ~R)

v = E3.e1
(R * v * ~R).show("R v ~R   (e1 → e2, i.e. 90° total)")

# Round-trip: read the rotation parameters back out of the multivector
print()
print("analyze(R) →", geo.analyze(R))


R = R2 * R1: 0.7071 - 0.7071 e12

R * ~R = 1


R v ~R   (e1 → e2, i.e. 90° total): e2


analyze(R) → Rotor(90.0° about Dir(0.00, 0.00, 1.00))


## 9. Pseudoscalar & the Hodge dual

The pseudoscalar $I = e_1 \wedge e_2 \wedge e_3$ squares to $-1$ in $G(3,0)$. The signed
dual $\star A = A \cdot I^{+}$ (`.dual()`) maps each blade to its orthogonal complement.
In E3 the dual-of-dual is $-A$, and the dual of a bivector is exactly the **cross
product**:

$$\star(a \wedge b) = a \times b$$


In [9]:
# The pseudoscalar is the full-volume element e1 ∧ e2 ∧ e3
(I3 * I3).show("I * I")
print()

# Signed dual ★A = A · I⁺  (Hodge dual); in E3 the dual-of-dual is −A
print("★e1  =", e1.dual(), "   ★e2  =", e2.dual(), "   ★e3  =", e3.dual())
print("★★e1 =", e1.dual().dual())
print()

# The cross product is the Hodge dual of the outer product:  a × b = ★(a ∧ b)
a = E3("1 e1 + 2 e2 + 3 e3")
b = E3("4 e1 + 5 e2 + 6 e3")

(a ^ b).show("a ^ b")
(a.op(b).dual()).show("★(a ^ b)  =  a × b")


I * I: - 1


★e1  = -e23    ★e2  = e13    ★e3  = -e12
★★e1 = -e1



a ^ b: - 3 e12 - 6 e13 - 3 e23

★(a ^ b)  =  a × b: - 3 e1 + 6 e2 - 3 e3

## 10. Bivectors and rotation planes

Every bivector corresponds to an **oriented plane**; its Hodge dual is that plane's
**normal** vector (the rotation axis). This is why the rotor about the $z$-axis is built
from the bivector $e_{12}$ — the $xy$-plane.


In [10]:
# A bivector is an oriented plane; its dual is the plane's normal (axis) vector
print("e12  ↔", geo.analyze(e12))
print("e23  ↔", geo.analyze(e23))
print("e31  ↔", geo.analyze(e31))
print()

# The dual maps a bivector (plane) to its normal (axis):
print("★e12 =", e12.dual(), "  (normal of the xy-plane is +z)")
print("★e23 =", e23.dual(), "  (normal of the yz-plane is +x)")
print("★e31 =", e31.dual(), "  (normal of the zx-plane is +y)")

# A rotor is scalar + bivector: the bivector fixes the rotation plane
R = geo(Rotor(angle=math.pi / 2, axis=Direction(0, 0, 1)))
R.show("Rotor about z  =  scalar part + e12 bivector part")


e12  ↔ Plane(pt=Point(0.00, 0.00, 0.00), n=Dir(0.00, 0.00, 1.00))
e23  ↔ Plane(pt=Point(0.00, 0.00, 0.00), n=Dir(1.00, 0.00, 0.00))
e31  ↔ Plane(pt=Point(0.00, 0.00, 0.00), n=Dir(0.00, -1.00, 0.00))

★e12 = e3   (normal of the xy-plane is +z)
★e23 = e1   (normal of the yz-plane is +x)
★e31 = e2   (normal of the zx-plane is +y)


Rotor about z  =  scalar part + e12 bivector part: 0.7071 - 0.7071 e12

## 11. Visual examples

`pytanga.viz` analyses multivectors on the way into the viewer (honouring the algebra's
`opns` flag): grade-1 vectors render as **arrows**, bivectors as **oriented planes**, and
rotors as their arc-plus-axis glyph. Three figures follow. Viewer setup and controls are
covered in [Part II — Visualization](../../visualization/).


In [11]:
from pytanga.viz import Visualizer

# Two vectors and the oriented plane they span
a = E3("2 e1")          # along x
b = E3("2 e2")          # along y
B = a ^ b               # bivector → the xy-plane through the origin

viz = Visualizer(title="E3 — vectors and their outer-product bivector")
viz.add(a, color="#ff4444", label="a")
viz.add(b, color="#4488ff", label="b")
viz.add(B, color="#44ff44", opacity=0.3, label="a ∧ b (oriented plane)")

viz.display_snapshot()   # renders the scene inline (serverless)


In [12]:
# The cross product is the Hodge dual of the outer product
a = E3("1 e1 + 2 e2 + 3 e3")
b = E3("4 e1 + 5 e2 + 6 e3")
c = a.op(b).dual()      # ★(a ∧ b) = a × b = (−3, 6, −3)

viz = Visualizer(title="E3 — cross product = Hodge dual of the outer product")
viz.add(a, color="#ff4444", label="a")
viz.add(b, color="#4488ff", label="b")
viz.add(a ^ b, color="#44ff44", opacity=0.25, label="a ∧ b (plane)")
viz.add(c, color="#ffcc00", label="★(a ∧ b) = a × b")

viz.display_snapshot()


In [13]:
# A rotor applied to a vector — before and after
v = E3.e1                                             # before
R = geo(Rotor(angle=math.pi / 2, axis=Direction(0, 0, 1)))
v_rot = R * v * ~R                                    # after → e2

viz = Visualizer(title="E3 — rotor applied to a vector (before / after)")
viz.add(v, color="#ff4444", label="v (before)")
viz.add(v_rot, color="#4488ff", label="R v ~R (after)")
viz.add(R, color="#ffcc00", label="Rotor (90° about z)")

viz.display_snapshot()


## 12. Summary & next steps

You now know how to work with Euclidean 3D multivectors:

| Concept | API |
|---|---|
| Vector / bivector from coordinates | `E3("…")` / `E3({…})` |
| Geometric product | `*` (`a.gp(b)`) |
| Outer product (bivector) | `^` (`a.op(b)`) |
| Inner product (dot) | `\|` (`a.ip(b)`) |
| Rotor (angle–axis) | `geo(Rotor(angle, Direction(...)))` |
| Apply rotor | `R * v * ~R` (`R.vp(v)`) |
| Pseudoscalar / Hodge dual | `E3.I` / `a.dual()` |
| Cross product | `a.op(b).dual()` |
| Bivector ↔ plane | `geo.analyze(bivector)` |

**Where to go next:**

- [**05 · Projective 3D (G(4,0))**](../05_projective_p3/) — adds a homogeneous
  coordinate for points and translations.
- [**08 · Duality & Complements**](../08_duality/) — the full picture of `dual()`,
  `ldual()`, and `complement()`.
